# Activation Clustering

Cluster-based analysis of model activations: identify activation archetypes,
transition patterns, position groupings, and component similarity.

References:
- Gurnee et al. (2023) "Finding Neurons in a Haystack"
- Bricken et al. (2023) "Towards Monosemanticity"

## Why This Matters

Activation clustering characterizes the patterns and statistics of internal model activations. Activation structure reveals how the model processes information — which components are active, how sparse the computation is, and what geometric patterns emerge.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_clustering import (
    residual_stream_clustering,
    layer_activation_archetypes,
    activation_transition_analysis,
    position_clustering,
    component_output_clustering,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Residual Stream Clustering

Cluster final-layer residual stream activations across multiple inputs.
Reveals groups of inputs that produce similar internal representations.

In [ ]:
tokens_list = [
    jnp.array([0, 5, 10, 15, 20]),
    jnp.array([1, 6, 11, 16, 21]),
    jnp.array([2, 7, 12, 17, 22]),
    jnp.array([40, 41, 42, 43, 44]),
    jnp.array([50, 51, 52, 53, 54]),
]
result = residual_stream_clustering(model, tokens_list, n_clusters=2)
print(f"Cluster assignments: {result['cluster_assignments']}")
print(f"Cluster sizes: {result['cluster_sizes']}")
print(f"Within-cluster variance: {result['within_cluster_variance']}")
print(f"Cluster center norms: {[np.linalg.norm(c):.4f for c in result['cluster_centers']]}")

## Layer Activation Archetypes

Find recurring activation patterns (archetypes) across all layers and positions.
Shows whether the model reuses similar activation patterns.

In [ ]:
result = layer_activation_archetypes(model, tokens, n_archetypes=3)
print(f"Archetype prevalence: {result['archetype_prevalence']}")
print(f"\nLayer-archetype distribution:")
for layer in range(cfg.n_layers):
    print(f"  Layer {layer}: {result['layer_archetype_distribution'][layer]}")
print(f"\nArchetype assignments (per layer x position):")
print(result['layer_archetype_assignments'])

## Activation Transition Analysis

Measure how activations change between consecutive layers: magnitude,
direction, and smoothness of transitions.

In [ ]:
result = activation_transition_analysis(model, tokens)
print(f"Transition magnitudes: {result['transition_magnitudes']}")
print(f"Mean transition magnitude: {result['mean_transition_magnitude']:.4f}")
print(f"Cosine continuity: {result['cosine_continuity']}")
print(f"Smoothness: {result['smoothness']:.4f}")

## Position Clustering

Cluster token positions by their activation patterns at a given layer.
Reveals which positions develop similar representations.

In [ ]:
result = position_clustering(model, tokens, layer=-1, n_clusters=2)
print(f"Position cluster assignments: {result['cluster_assignments']}")
print(f"Cluster sizes: {result['cluster_sizes']}")
print(f"\nPosition similarity matrix:")
np.set_printoptions(precision=2)
print(result['position_similarity'])

## Component Output Clustering

Group model components (attention layers, MLPs) by the similarity of their
outputs. Reveals which components produce similar residual stream updates.

In [ ]:
result = component_output_clustering(model, tokens, n_clusters=2)
print(f"Components: {result['component_names']}")
print(f"Cluster assignments: {result['cluster_assignments']}")
print(f"\nComponent similarity matrix:")
print(np.array2string(result['similarity_matrix'], precision=2))